# Sets, fields, and distributions

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/tutorial/core/03_sets_fields_distributions.ipynb)

Minilink uses three distinct mathematical objects for regions, scalar functions, and uncertainty:
- **`Set`** and **`InputSet`**: deterministic feasibility regions $z \in Z(t)$ and $u \in U(x, t)$ defined by nonnegative margins $\mathrm{margin} \ge 0$;
- **`Field`**: scalar functions $V(x, u, t) \in \mathbb{R}$ (such as energy or Lyapunov candidates) and their gradients $\nabla_x V$;
- **`Distribution`**: probability laws $z \sim p$ with `.sample(key, n)`, `.mean`, `.cov`, and optional `.support`.

In [ ]:
# Local conda: minilink already installed. Colab: clone + path + meshcat.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")
    get_ipython().system("pip install -q meshcat")

## 1. `Set`: allowable regions of a vector

A `Set` is its margin: $z \in Z(t) \iff \mathrm{margin}(z, t) \ge 0$ componentwise. A positive margin is inside, zero is on the boundary, negative is outside.

| Call | Returns | Meaning |
| --- | --- | --- |
| `S.params` | `dict` | The set's numbers, edited in place: `S.params["upper"] = ...` |
| `S.margin(z, t=0.0, params=None)` | `array (k,)` | Feasibility margins; the one method a subclass writes |
| `S.contains(z, t=0.0, params=None)` | boolean (on `z`'s array module) | Whether every margin is $\ge 0$ |
| `S.sample(key, n=None, params=None)` | `array (dim,)`, or `(n, dim)` with `n` | Draw from the set, where the set supports it; `key` is an integer seed, a NumPy generator or a JAX key (JAX arrays out) |
| `S.bounding_box()` | `BoxSet \| None` | The tightest axis-aligned box containing the set, `None` when it has none |
| `S.as_cost(*, weight=1.0, shaping=None)` | `CostFunction` | The running cost of leaving the set, $g = w \cdot \mathrm{shaping}(\min_i m_i(x))$; `shaping=None` is the quadratic hinge $\max(-v, 0)^2$, zero inside |
| `S1 & S2` | `IntersectionSet` | The intersection: the members' margins concatenated |

The concrete sets:

| Class | Region | `params` | `margin` shape | `sample` | `bounding_box()` |
| --- | --- | --- | --- | --- | --- |
| `BoxSet(lower, upper)` | $\text{lower} \le z \le \text{upper}$ | `{"lower", "upper"}` | `(2n,)`: $[z - \text{lower};\; \text{upper} - z]$ | uniform (finite bounds) | itself |
| `BallSet(center, radius)` | $\lVert z - c \rVert_2 \le r$ | `{"center", "radius"}` | `(1,)`: $r - \lVert z - c \rVert_2$ | uniform | $c \pm r$ |
| `SingletonSet(point)` | $z = p$ | `{"point"}` | `(n,)`: $-\lvert z - p \rvert$ | the point | the point |
| `EllipsoidSet(field, lower=None, upper=None)`, `EllipsoidSet.from_matrix(center, P, level)` | $(z - c)^\top P (z - c) \le \text{level}$ | `{"center", "P", "level"}` | `(1,)` | uniform | $c \pm \sqrt{\text{level} \cdot \mathrm{diag}(P^{-1})}$ |
| `FieldSet(field, lower=0.0, upper=None)` | $\text{lower} \le V(z) \le \text{upper}$ | `{"field", "lower", "upper"}` (the bounds that are set) | one entry per bound | — | `None` |
| `CallableSet(margin, params=None)` | `margin(z, t, params) >= 0` | as given | the function's | — | `None` |
| `IntersectionSet(sets)`, `IntersectionSet.of(*sets)` | every member | the members' trees by class name | the members', concatenated | — | the members' boxes, intersected |

`BoxSet`, `BallSet`, `SingletonSet`, `FieldSet` and `IntersectionSet` also state `S.dim`; the named numbers (`box.lower`, `ball.center`, `ball.radius`, `goal_point.point`) are views of `params`. `SingletonSet` adds `residual(z, params=None)`, $z - p$ of shape `(n,)`: the equality residual an optimizer reads.

In [ ]:
import numpy as np

from minilink import BallSet, BoxSet, Pendulum, SingletonSet

X = BoxSet([-np.pi, -4.0], [np.pi, 4.0])
z_out = np.array([1.0, 5.0])

X.dim, X.params

In [ ]:
X.margin(z_out), X.contains(z_out)

In [ ]:
z_in = np.array([1.0, 2.0])
X.margin(z_in), X.contains(z_in)

`X.sample(key)` returns one point of shape `(dim,)`; with `n` it returns `(n, dim)`:

In [ ]:
single_point = X.sample(0)
three_points = X.sample(0, n=3)
single_point.shape, three_points.shape, three_points

A system declares its state box on `plant.state.box` and its input box on `plant.input_box`:

In [ ]:
plant = Pendulum()
plant.state.box, plant.input_box

In [ ]:
goal_ball = BallSet(center=[np.pi, 0.0], radius=0.2)
goal_ball.margin(np.array([np.pi + 0.05, 0.0])), goal_ball.bounding_box()

In [ ]:
goal_point = SingletonSet([np.pi, 0.0])
z_test = np.array([3.0, 0.1])
goal_point.residual(z_test), goal_point.margin(z_test), goal_point.contains(np.array([np.pi, 0.0]))

In [ ]:
safe_goal = X & goal_ball
safe_goal.margin(np.array([np.pi, 0.0])).shape, list(safe_goal.params), safe_goal.bounding_box()

### Writing a `Set`

A set subclasses `Set`, sets `self.params`, and writes `margin`; `contains` and `as_cost` follow from it.

In [ ]:
from minilink.core import Set


class HalfspaceSet(Set):
    def __init__(self, limit=1.0):
        self.params = {"limit": float(limit)}

    def margin(self, z, t=0.0, params=None):
        params = self.params if params is None else params

        limit = params["limit"]

        # z₀ ≤ limit
        m = limit - z[0:1]

        return m


halfspace = HalfspaceSet(limit=1.0)
halfspace.margin(np.array([0.4, 0.0])), halfspace.contains(np.array([1.5, 0.0]))

## 2. `InputSet`: admissible inputs $u \in U(x, t)$

An `InputSet` is the input counterpart of a `Set`: its member is `u`, and the margin may depend on the state `x`.

| Call | Returns | Meaning |
| --- | --- | --- |
| `U.margin(u, x=None, t=0.0, params=None)` | `array (k,)` | Feasibility margins of `u` at `(x, t)` |
| `U.contains(u, x=None, t=0.0, params=None)` | boolean | Whether `u` is admissible at `(x, t)` |
| `U.sample(key, n=None, x=None, t=0.0, params=None)` | `array (m,)`, or `(n, m)` with `n` | Draw admissible inputs, where supported |
| `U.bounding_box()` | `BoxSet \| None` | The tightest box of inputs containing the set |

| Class | Region | `params` |
| --- | --- | --- |
| `BoxInputSet(box)` | $\text{lower} \le u \le \text{upper}$, independent of the state; `box` is a `BoxSet`, kept as `U.box`, with `U.lower`, `U.upper`, `U.dim` | its box's `{"lower", "upper"}` |
| `FieldInputSet(field, lower=0.0, upper=None)` | $\text{lower} \le V(x, u, t) \le \text{upper}$, built by `V.as_input_set(...)` | `{"field", "lower", "upper"}` |

`sys.input_box` is the `BoxInputSet` of a system's input ports. `PlanningProblem(U=...)` takes an `InputSet`, or a `BoxSet` that it wraps.

In [ ]:
from minilink import BoxInputSet

U = BoxInputSet(BoxSet([-5.0], [5.0]))
U.margin(np.array([3.0])), U.contains(np.array([6.0]))

In [ ]:
U.sample(0, n=3), U.bounding_box()

## 3. `Field`: scalar functions $V(x, u, t)$

A `Field` is a scalar function on the model's arguments; it reads the ones it depends on and ignores the rest.

| Call | Returns | Meaning |
| --- | --- | --- |
| `V.params` | `dict` | The field's numbers |
| `V.value(x, u=None, t=0.0, params=None)` | scalar | $V(x, u, t)$; the one method a subclass writes |
| `V(x, u=None, t=0.0, params=None)` | scalar | The field called, the same as `value`: `V(x)`, the time by keyword, `V(x, t=t)` |
| `V.gradient(x, u=None, t=0.0, params=None)` | `array (n,)` | $\partial V / \partial x$: exact where the class states it, else `jax.grad` on JAX arrays and central differences on NumPy |
| `V.as_set(*, lower=UNSET, upper=None)` | `Set` | The states where $\text{lower} \le V(x) \le \text{upper}$, each bound when given (`None` drops it): `upper` alone is the sublevel set, and with no bound given the set is $V(x) \ge 0$ |
| `V.as_input_set(*, lower=UNSET, upper=None)` | `InputSet` | The inputs where $\text{lower} \le V(x, u) \le \text{upper}$ at the state `x`, the same rule |
| `V.as_cost(*, weight=1.0, shaping=None)` | `CostFunction` | Running cost $g = w \cdot \mathrm{shaping}(V(x, u, t))$, no terminal cost; `shaping=None` is $V$ itself |
| `V.as_terminal_cost(*, weight=1.0, shaping=None)` | `CostFunction` | Terminal cost $h = w \cdot \mathrm{shaping}(V(x, t))$, no running cost |

The concrete fields:

| Class | Field | `params` | Own attributes |
| --- | --- | --- | --- |
| `QuadraticField(center, P, t=None)` | $V(x, t) = (x - c)^\top P(t)\,(x - c)$; `P` is one matrix `(n, n)`, or a schedule `(N, n, n)` on the times `t`, read linearly and held past its ends | `{"center", "P"}`, and `"t"` for a schedule | `V.center`, `V.P`, `V.t`, `V.dim`, `V.matrix_at(t=0.0, params=None)`; `as_set(upper=level)` of a constant `P` is an `EllipsoidSet` |
| `GridField(levels, values, t=None, *, method="linear", outside=None)` | A table on a regular grid, read in the cell holding `x`: `method` is `"linear"`, `"nearest"`, or `"cubic"` and `"quintic"` (NumPy only); outside the grid the edge is held, or $V$ is `outside` when given | `{"levels", "values"}`, and `"t"`, `"outside"` when given | `table.dim`, `table.t`, `table.method`, `table.table_at(t=0.0, params=None)` |
| `MarginField(set_)` | $V(x, t) = \min_i m_i(x, t)$, a set's least margin | the set's | `margin_field.dim` |

The shaping of `as_cost` is a penalty of the field's value $v$, from `minilink.core`: `quadratic_hinge(threshold=0.0)` is $\max(\text{threshold} - v, 0)^2$, `quadratic_excess(threshold=0.0)` is $\max(v - \text{threshold}, 0)^2$, `inverse_barrier(epsilon=1e-2)` is $1 / \max(v, \epsilon)^2$, and `occupancy(scale=1.0)` is $\mathrm{sigmoid}(-v / \text{scale})$.

In [ ]:
from minilink import GridField, QuadraticField
from minilink.core import MarginField

V = QuadraticField(np.zeros(2), np.diag([2.0, 0.5]))
x_query = np.array([1.0, 2.0])

V(x_query), V.value(x_query), V.gradient(x_query)

In [ ]:
E = V.as_set(upper=1.0)
shell = V.as_set(lower=1.0, upper=4.0)
E, E.contains(x_query), E.bounding_box(), type(shell).__name__, shell.contains(x_query)

In [ ]:
running_cost = V.as_cost(weight=2.0)
terminal_cost = V.as_terminal_cost(weight=5.0)
wall_penalty = X.as_cost(weight=10.0)

running_cost.g(x_query, np.zeros(1)), terminal_cost.h(x_query), wall_penalty.g(z_out, np.zeros(1))

In [ ]:
margin_field = MarginField(X)
margin_field.value(z_in), margin_field.gradient(z_in)

In [ ]:
nodes = np.linspace(-1.0, 1.0, 5)
table = GridField((nodes, nodes), nodes[:, None] ** 2 + nodes[None, :] ** 2, outside=10.0)
table(np.array([0.5, 0.5])), table.gradient(np.array([0.25, 0.5])), table(np.array([2.0, 0.0]))

The ellipsoid `E` intersected with a ball, drawn by membership of points sampled in `X`:

In [ ]:
import matplotlib.pyplot as plt

points = X.sample(0, 2000)
ball = BallSet([0.5, 0.0], 1.2)
E_ball = E & ball
inside = np.array([E_ball.contains(point) for point in points])

fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(*points[~inside].T, s=4, c="lightgray", label="X only")
ax.scatter(*points[inside].T, s=4, c="tab:blue", label="E & ball")
ax.set_xlabel("theta")
ax.set_ylabel("dtheta")
ax.legend()
plt.show()

## 4. `Distribution`: probability laws over vectors

A `Distribution` is a random vector $z \sim p$. A `Set` answers membership; a `Distribution` answers draws and moments, and states its bounded support when it has one.

| Call | Returns | Meaning |
| --- | --- | --- |
| `law.dim` | `int` | Dimension $d$ of a draw |
| `law.mean` | `ndarray (d,)` | $\mathbb{E}[z]$ |
| `law.cov` | `ndarray (d, d)` | $\mathrm{Cov}(z)$ |
| `law.support` | `Set \| None` | The bounded support, `None` when the law has none |
| `law.params` | `dict` | The law's numbers, edited in place: `law.params["mean"] = ...` |
| `law.sample(key, n=None, params=None)` | `array (d,)`, or `(n, d)` with `n` | Draw; `key` is an integer seed, a NumPy generator or a JAX key (JAX arrays out, traceable); `key=None` draws nothing and returns the mean at `params` |

| Class | Law | `params` | `support` | Own attributes |
| --- | --- | --- | --- | --- |
| `Gaussian(mean, std=None, cov=None)` | Normal, independent components (`std`) or a full covariance (`cov`), one of the two | `{"mean", "std"}` or `{"mean", "cov"}` | `None` | `start_law.std` |
| `Uniform(lower, upper)` | Uniform on a box | `{"lower", "upper"}` | the `BoxSet` | `mass_law.lower`, `mass_law.upper` |
| `Particles(points)` | Equal weights on the rows of `points` `(N, d)` | `{"points"}` | `None` | `cloud.points` |

In [ ]:
from minilink import Gaussian, Uniform
from minilink.core import Particles

start_law = Gaussian(mean=[0.0, 0.0], std=[0.1, 0.5])
start_law.dim, start_law.mean, start_law.cov, start_law.support

In [ ]:
# One draw, three draws, and the mean (key=None)
start_law.sample(0), start_law.sample(0, n=3), start_law.sample(None)

In [ ]:
mass_law = Uniform(lower=[0.8], upper=[1.2])
mass_law.mean, mass_law.cov, mass_law.support

In [ ]:
cloud = Particles(np.array([[0.0, 0.0], [1.0, 1.0], [2.0, 0.0]]))
cloud.dim, cloud.mean, cloud.cov, cloud.sample(0, n=4)

## Summary

| Abstraction | Concrete classes | Equation and main methods | Role in a `PlanningProblem` |
| --- | --- | --- | --- |
| `Set` | `BoxSet`, `BallSet`, `SingletonSet`, `EllipsoidSet`, `FieldSet`, `CallableSet`, `IntersectionSet` | `margin(z, t)`, `contains(z, t)`, `sample(key, n)`, `bounding_box()`, `as_cost()` | The state sets `X`, `X0`, `Xf` |
| `InputSet` | `BoxInputSet`, `FieldInputSet` | `margin(u, x, t)`, `contains(u, x, t)`, `sample(key, n)`, `bounding_box()` | The input set `U` |
| `Field` | `QuadraticField`, `GridField`, `MarginField` | `value(x, u, t)`, `gradient(x, u, t)`, `as_set()`, `as_input_set()`, `as_cost()`, `as_terminal_cost()` | Level sets, penalties, and a planner's cost-to-go |
| `Distribution` | `Gaussian`, `Uniform`, `Particles` | `sample(key, n)`, `mean`, `cov`, `support` | The start law `x0` and the laws of `distributions` |

Next, [`04_costs`](04_costs.ipynb): running costs, terminal costs, stage costs, and cost algebra.